# PyTorch segment / block
실제 CUDA 실행 · H100 PCIe · native allocator · 2026-10-16

4 MiB와 6 MiB 텐서가 같은 segment에 들어가는지, 해제한 block을 재사용하는지 확인한다.

In [ ]:
import gc
import json
from pprint import pprint

import torch

MIB = 1024 * 1024
snapshots = []
torch.cuda.init()
print(torch.__version__)
print(torch.cuda.get_device_name())
print(torch.cuda.get_allocator_backend())

`memory_snapshot()`은 segment의 목록을 반환한다. 아래 함수는 원본 snapshot을 보관하고, 추적 중인 segment의 크기와 block 상태만 MiB 단위로 출력한다.

In [ ]:
def inspect_segment(address):
    snapshot = torch.cuda.memory_snapshot()
    snapshots.append(snapshot)
    pprint([
        {
            "total_mib": segment["total_size"] // MIB,
            "allocated_mib": segment["allocated_size"] // MIB,
            "blocks": [
                (block["size"] // MIB, block["state"])
                for block in segment["blocks"]
            ],
        }
        for segment in snapshot
        if segment["address"] == address
    ], sort_dicts=False)

## 1. 텐서는 10 MiB인데 segment는 얼마나 받았을까

In [ ]:
a = torch.empty(4 * MIB, dtype=torch.uint8, device="cuda")
b = torch.empty(6 * MIB, dtype=torch.uint8, device="cuda")
segment_address = a.data_ptr()
inspect_segment(segment_address)

## 2. a를 놓으면 4 MiB block은 어디로 갈까

In [ ]:
del a
gc.collect()
torch.cuda.synchronize()
inspect_segment(segment_address)

## 3. 새 3 MiB 텐서는 빈 block을 재사용할까

In [ ]:
c = torch.empty(3 * MIB, dtype=torch.uint8, device="cuda")
print("tensor_mib:", c.numel() * c.element_size() // MIB)
print("reused_a_address:", c.data_ptr() == segment_address)
inspect_segment(segment_address)

## 4. 남은 텐서도 놓으면 block은 어떻게 합쳐질까

In [ ]:
del b, c
gc.collect()
torch.cuda.synchronize()
inspect_segment(segment_address)

## 5. empty_cache() 뒤에도 segment가 남아 있을까

In [ ]:
torch.cuda.empty_cache()
inspect_segment(segment_address)

In [ ]:
with open("/tmp/pytorch-segment-snapshots.json", "w") as file:
    json.dump(snapshots, file, indent=2)